<!-- CODEx bilingual cell explanation: start -->
### Cell 01 — 环境配置与全局参数 / Environment setup and global parameters

**中文说明**：导入数值计算、数据处理、SQLite、EnergyPlus 调用和绘图所需模块，建立项目目录结构，并集中定义采样规模、随机种子、EnergyPlus 路径、气象文件、生活热水和可行性筛选阈值。该 cell 是整个参数化仿真数据库的运行入口；后续所有采样、IDF 生成、仿真和后处理均读取这里的 CONFIG。为便于本地验证，代码支持通过环境变量切换小样本快速模式和是否真正运行 EnergyPlus。

**输入与依赖**：依赖本地 Python/Jupyter 环境、项目根目录和后续单元需要共享的配置参数。

**主要输出**：建立路径、随机种子、绘图样式、配置字典或输出目录等基础对象。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Imports the numerical, tabular, SQLite, EnergyPlus subprocess, and plotting dependencies, creates the project directory structure, and defines the central CONFIG object for sample size, random seed, EnergyPlus paths, weather data, domestic hot-water assumptions, and feasibility bounds. This cell is the execution entry point for the parametric simulation database; all downstream sampling, IDF generation, simulation, and post-processing read from this CONFIG. Environment variables are supported so that local smoke tests can run without launching the full EnergyPlus batch.

**Inputs and dependencies**: Depends on the local Python/Jupyter environment, the project root, and shared configuration values used by later cells.

**Main outputs**: Creates paths, random seeds, plotting defaults, configuration dictionaries, or output directories.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:

import math
import os
import sqlite3
import subprocess
import textwrap
from pathlib import Path
import sys
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "grid.linestyle": "--",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "input" / "Beijing.epw").exists() or (candidate / "data" / "step1_simulation_dataset.csv").exists():
            return candidate
        if (candidate / "AGENTS.md").exists() and (candidate / "outputs_step1").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
CODE_DIR = PROJECT_ROOT / "experiment_code"
DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = PROJECT_ROOT / "input"
OUTPUT_DIR = PROJECT_ROOT / "outputs_step1"
IDF_DIR = OUTPUT_DIR / "generated_idf"
RUN_DIR = OUTPUT_DIR / "runs"
FIG_DIR = OUTPUT_DIR / "figures"
PAPER_ASSETS_DIR = PROJECT_ROOT / "paper_assets"
PAPER_ASSETS_FIG_DIR = PAPER_ASSETS_DIR / "figures"
for search_path in [PROJECT_ROOT, CODE_DIR]:
    if search_path.exists() and str(search_path) not in sys.path:
        sys.path.insert(0, str(search_path))

for p in [DATA_DIR, INPUT_DIR, OUTPUT_DIR, IDF_DIR, RUN_DIR, FIG_DIR, PAPER_ASSETS_DIR, PAPER_ASSETS_FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def save_figure(figure, path, **kwargs):
    """Save figures through a temporary file to avoid interrupted overwrites on Windows/Jupyter."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(f"{path.stem}.__tmp__{path.suffix}")
    figure.savefig(tmp_path, **kwargs)
    tmp_path.replace(path)

CONFIG = {
    # Usually only the sample size needs to be changed.
    "n_samples": int(os.environ.get("EUI_N_SAMPLES", "20000")),
    "random_seed": 42,
    # EnergyPlus 25.2.0 path (Windows example).
    "energyplus_exe": r"C:/EnergyPlusV25-2-0/energyplus.exe",
    "expandobjects_exe": r"C:\EnergyPlusV25-2-0\ExpandObjects.exe",
    # Beijing weather file path.
    "weather_epw": str(INPUT_DIR / "Beijing.epw"),
    # Whether to actually run EnergyPlus simulations.
    "run_energyplus": os.environ.get("EUI_RUN_ENERGYPLUS", "0") == "1",
    "timeout_seconds": int(os.environ.get("EUI_TIMEOUT_SECONDS", "900")),
    "clean_idf_dir": os.environ.get("EUI_CLEAN_IDF_DIR", "0") == "1",
    "clean_run_dir": os.environ.get("EUI_CLEAN_RUN_DIR", "0") == "1",
    # Engineering post-processing parameters.
    "dhw_cold_water_temp_c": 15.0,
    "dhw_density_kg_m3": 1000.0,
    "water_cp_kj_kgk": 4.186,
    "fan_delta_p_pa": 600.0,
    # Controls for feasible area ratios.
    "min_usable_ratio": 0.55,
    "max_usable_ratio": 0.95,
}

print(CONFIG)

<!-- CODEx bilingual cell explanation: start -->
### Cell 02 — 本地路径与输入文件检查 / Local path and input-file check

**中文说明**：打印 Python 解释器、当前工作目录、EnergyPlus 可执行文件、ExpandObjects 和 Beijing.epw 的实际路径及存在性，帮助作者在运行正式仿真前发现路径配置错误。该检查不改变数据，只给出可复现性诊断信息。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Prints the Python executable, working directory, EnergyPlus executable, ExpandObjects, and Beijing.epw paths with existence checks. It helps identify local configuration errors before the full simulation batch is launched. The cell is diagnostic only and does not mutate the dataset.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
from pathlib import Path
import os, sys

print("Current Python executable: ", sys.executable)
print("Current working directory: ", Path.cwd())

ep = Path(CONFIG["energyplus_exe"])
wth = Path(CONFIG["weather_epw"])

print("\n[EnergyPlus path]")
print("Original value: ", CONFIG["energyplus_exe"])
print("Resolved path: ", ep)
print("Exists: ", ep.exists())
print("Is file: ", ep.is_file())

print("\n[Weather file path]")
print("Original value: ", CONFIG["weather_epw"])
print("Resolved path: ", wth)
print("Exists: ", wth.exists())
print("Is file: ", wth.is_file())

print("\n[Files under the input directory]")
input_dir = INPUT_DIR
print("Input directory exists: ", input_dir.exists())
if input_dir.exists():
    for f in input_dir.iterdir():
        print(" -", f.name)

<!-- CODEx bilingual cell explanation: start -->
### Cell 03 — 参数空间定义 / Parameter-space definition

**中文说明**：以结构化表格定义酒店建筑的围护结构、几何形态、功能空间、人员与系统运行参数的采样范围、变量类型和默认值。该表是 LHS 抽样、IDF 生成、敏感性分析和机器学习建模的共同输入，保证论文表格与代码变量一致。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Defines the sampled hotel-building parameter space as a structured table covering envelope, geometry, functional area, occupancy, and system-operation variables. This table is the common input to LHS sampling, IDF generation, sensitivity analysis, and machine-learning modelling, keeping manuscript tables and code variables aligned.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 1) Parameter-space definition ----------
# Note: some variables in the manuscript are naturally coupled or redundant:
# - building_length / building_width / aspect_ratio
# - insulation thickness / wall U-value
# - roof insulation thickness / roof U-value
# The fields remain compatible with the manuscript, but sampling and generation use independent variables, derived variables, and constraint checks.

FEATURES = [
    {"name": "insul_thick", "kind": "float", "min": 0.05, "max": 0.12, "default": 0.08},
    {"name": "wwr", "kind": "float", "min": 0.25, "max": 0.60, "default": 0.40},
    {"name": "wall_thick", "kind": "float", "min": 0.20, "max": 0.30, "default": 0.24},
    {"name": "u_win_n", "kind": "float", "min": 0.8, "max": 2.0, "default": 1.4},
    {"name": "u_win_s", "kind": "float", "min": 0.8, "max": 2.0, "default": 1.4},
    {"name": "u_win_e", "kind": "float", "min": 0.8, "max": 2.0, "default": 1.4},
    {"name": "u_win_w", "kind": "float", "min": 0.8, "max": 2.0, "default": 1.4},
    {"name": "u_wall", "kind": "float", "min": 0.25, "max": 0.80, "default": 0.45},
    {"name": "u_roof", "kind": "float", "min": 0.20, "max": 0.45, "default": 0.30},
    {"name": "u_ground", "kind": "float", "min": 0.15, "max": 0.40, "default": 0.35},
    {"name": "shgc_n", "kind": "float", "min": 0.20, "max": 0.65, "default": 0.40},
    {"name": "shgc_s", "kind": "float", "min": 0.20, "max": 0.45, "default": 0.35},
    {"name": "shgc_e", "kind": "float", "min": 0.20, "max": 0.65, "default": 0.40},
    {"name": "shgc_w", "kind": "float", "min": 0.20, "max": 0.65, "default": 0.40},
    {"name": "window_type_id", "kind": "int", "min": 1, "max": 3, "default": 2},
    {"name": "roof_insul_thick", "kind": "float", "min": 0.08, "max": 0.15, "default": 0.12},
    {"name": "floor_num", "kind": "int", "min": 6, "max": 20, "default": 10},
    {"name": "public_area", "kind": "float", "min": 80, "max": 150, "default": 110},
    {"name": "room_area", "kind": "float", "min": 22, "max": 30, "default": 26},
    {"name": "room_count", "kind": "int", "min": 36, "max": 240, "default": 120},
    {"name": "building_length", "kind": "float", "min": 35, "max": 50, "default": 42},
    {"name": "building_width", "kind": "float", "min": 12, "max": 18, "default": 15},
    {"name": "aspect_ratio", "kind": "float", "min": 2.0, "max": 3.0, "default": 2.5},
    {"name": "floor_height", "kind": "float", "min": 2.8, "max": 3.3, "default": 3.0},
    {"name": "orientation_deg", "kind": "float", "min": 0, "max": 90, "default": 0},
    {"name": "equip_power", "kind": "float", "min": 2.5, "max": 4.0, "default": 3.2},
    {"name": "dhw_per_person", "kind": "float", "min": 0.06, "max": 0.20, "default": 0.12},
    {"name": "occupancy_density", "kind": "float", "min": 0.08, "max": 0.25, "default": 0.12},
    {"name": "light_power", "kind": "float", "min": 4.0, "max": 7.0, "default": 5.5},
    {"name": "cool_set", "kind": "float", "min": 24, "max": 26, "default": 25},
    {"name": "heat_set", "kind": "float", "min": 19, "max": 22, "default": 20},
    {"name": "dhw_temp", "kind": "float", "min": 45, "max": 55, "default": 50},
    {"name": "cop_cooling", "kind": "float", "min": 3.0, "max": 5.0, "default": 4.0},
    {"name": "cop_heating", "kind": "float", "min": 2.2, "max": 4.2, "default": 3.0},
    {"name": "boiler_eff", "kind": "float", "min": 0.82, "max": 0.95, "default": 0.90},
    {"name": "fan_eff", "kind": "float", "min": 0.55, "max": 0.70, "default": 0.65},
    {"name": "fresh_air_ach", "kind": "float", "min": 0.5, "max": 1.2, "default": 0.8},
    {"name": "operation_hours", "kind": "float", "min": 2000, "max": 3000, "default": 2600},
]

feature_df = pd.DataFrame(FEATURES)
feature_df.head(38)

<!-- CODEx bilingual cell explanation: start -->
### Cell 04 — LHS 抽样与约束派生 / LHS sampling and dependency resolution

**中文说明**：使用 scipy.stats.qmc 的 Latin Hypercube Sampling 生成高维参数组合，并通过几何派生关系计算建筑宽度、总建筑面积、体积、可用面积比、估计人数、运行时段比例和等效热阻。筛选逻辑保留原始 LHS 索引，使后续筛选前后分布比较可准确追踪每个样本。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Generates high-dimensional parameter combinations with scipy.stats.qmc Latin Hypercube Sampling and resolves derived geometry, gross floor area, volume, usable-area ratio, estimated occupancy, schedule fraction, and equivalent thermal resistance. The filtering logic preserves the original LHS index so that pre/post-screening distribution diagnostics can correctly track retained and rejected samples.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 2) LHS sampling and constraint correction ----------

def lhs_sample(feature_table: pd.DataFrame, n_samples: int, seed: int = 42) -> pd.DataFrame:
    active = feature_table.reset_index(drop=True)
    sampler = qmc.LatinHypercube(d=len(active), seed=seed)
    unit = sampler.random(n=n_samples)

    out = {}
    for i, row in active.iterrows():
        lo, hi = float(row["min"]), float(row["max"])
        values = lo + (hi - lo) * unit[:, i]
        if row["kind"] == "int":
            values = np.rint(values).astype(int)
            values = np.clip(values, int(lo), int(hi))
        out[row["name"]] = values

    df = pd.DataFrame(out)
    df["sample_id"] = [f"sample_{i:04d}" for i in range(1, len(df) + 1)]
    return df


def resolve_dependencies(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Length, width, and aspect ratio: length plus aspect ratio control the geometry; width is derived to avoid inconsistent inputs.
    df["building_width"] = (df["building_length"] / df["aspect_ratio"]).clip(12, 18)

    # Building geometry metrics.
    df["gross_floor_area_m2"] = df["building_length"] * df["building_width"] * df["floor_num"]
    df["footprint_area_m2"] = df["building_length"] * df["building_width"]
    df["building_height_m"] = df["floor_num"] * df["floor_height"]
    df["building_volume_m3"] = df["gross_floor_area_m2"] * df["floor_height"]

    # Functional-area check: guest-room area plus public area should not exceed the gross floor area unrealistically.
    df["guestroom_total_area_m2"] = df["room_area"] * df["room_count"]
    df["usable_area_m2"] = df["guestroom_total_area_m2"] + df["public_area"]
    df["usable_area_ratio"] = df["usable_area_m2"] / df["gross_floor_area_m2"]
    df["geometry_feasible"] = df["usable_area_ratio"].between(CONFIG["min_usable_ratio"], CONFIG["max_usable_ratio"])

    # Occupancy estimate: use the more conservative value from room-based and area-density estimates.
    guest_based_people = df["room_count"] * 1.6
    density_based_people = df["gross_floor_area_m2"] * df["occupancy_density"]
    df["estimated_people"] = np.maximum(1.0, np.minimum(guest_based_people, density_based_people))

    # Convert annual operating hours into average daily operating hours for schedules.
    df["daily_operation_hours"] = (df["operation_hours"] / 365.0).clip(4, 24)
    df["schedule_on_fraction"] = (df["daily_operation_hours"] / 24.0).clip(0.15, 1.0)

    # Simplified envelope thermal resistance: U-values control the model; thickness variables are retained in the dataset but do not double-control construction performance.
    # Approximation used here: R_layer ~= 1/U - R_films; small values are clipped to avoid negative resistance.
    df["wall_R_m2K_W"] = (1.0 / df["u_wall"] - 0.17).clip(0.15, None)
    df["roof_R_m2K_W"] = (1.0 / df["u_roof"] - 0.14).clip(0.15, None)
    df["floor_R_m2K_W"] = (1.0 / df["u_ground"] - 0.17).clip(0.15, None)

    # Prototype visible transmittance is mapped from window_type_id.
    vt_map = {1: 0.65, 2: 0.55, 3: 0.45}
    df["glass_vt"] = df["window_type_id"].map(vt_map).fillna(0.65)

    return df

samples_raw = lhs_sample(feature_df, CONFIG["n_samples"], CONFIG["random_seed"])
samples_raw["source_lhs_index"] = np.arange(len(samples_raw))
samples_all = resolve_dependencies(samples_raw)
samples = samples_all.loc[samples_all["geometry_feasible"]].copy().reset_index(drop=True)

print(f"Initial number of samples: {len(samples_raw)}")
print(f"Samples retained after geometry/function filters: {len(samples)}")
samples.head(30)

<!-- CODEx bilingual cell explanation: start -->
### Cell 05 — 可行性筛选分布诊断 / Feasibility-screening distribution diagnostics

**中文说明**：绘制筛选前可用面积比直方图、筛选前后密度对比和关键参数二维覆盖图，并输出每个输入变量筛选前后的 KS 统计量与 Jensen-Shannon 距离。该 cell 定量检查可行性筛选对参数空间覆盖的影响。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots the pre-filter usable-area-ratio histogram, the pre/post-filter density comparison, and a two-dimensional coverage check for key parameters. It also exports per-variable Kolmogorov-Smirnov statistics and Jensen-Shannon distances between pre- and post-filter distributions. It quantifies how feasibility screening changes parameter-space coverage and potential selection bias.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ============================================================
# Feasibility-screening distribution analysis
# Quantifies pre/post-filter shifts and visualises retained coverage.
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp

rng = np.random.default_rng(CONFIG["random_seed"])

# Use the resolved full sample table so retained/rejected samples are tracked
# by the original LHS row rather than by a reset index.
all_resolved = samples_all.copy()
raw_ratio = all_resolved["usable_area_ratio"].copy()
retained_mask = all_resolved["geometry_feasible"].to_numpy()
filtered_ratio = all_resolved.loc[retained_mask, "usable_area_ratio"].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)

# Panel 1: Raw ratio histogram with filter bounds.
ax = axes[0]
ax.hist(raw_ratio, bins=80, color='steelblue', edgecolor='white', alpha=0.8,
        label=f'全部 LHS 样本 (n={len(raw_ratio):,})')
ax.axvline(CONFIG["min_usable_ratio"], color='red', linestyle='--', linewidth=2,
           label=f'下界 {CONFIG["min_usable_ratio"]}')
ax.axvline(CONFIG["max_usable_ratio"], color='darkred', linestyle='--', linewidth=2,
           label=f'上界 {CONFIG["max_usable_ratio"]}')
retention = len(filtered_ratio) / len(raw_ratio) * 100
ax.set_xlabel('可用面积/总建筑面积比')
ax.set_ylabel('样本数')
ax.set_title(f'筛选前分布 | 保留率 {retention:.1f}% ({len(filtered_ratio):,}/{len(raw_ratio):,})')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Panel 2: Pre vs post density comparison.
ax = axes[1]
ax.hist(raw_ratio, bins=80, color='steelblue', edgecolor='white', alpha=0.5,
        density=True, label=f'筛选前 (n={len(raw_ratio):,})')
ax.hist(filtered_ratio, bins=40, color='darkorange', edgecolor='white', alpha=0.7,
        density=True, label=f'筛选后 (n={len(filtered_ratio):,})')
ax.set_xlabel('可用面积/总建筑面积比')
ax.set_ylabel('密度')
ax.set_title('筛选前后分布对比（归一化）')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Panel 3: 2D parameter coverage check using true retained/rejected masks.
ax = axes[2]
rejected_index = np.flatnonzero(~retained_mask)
retained_index = np.flatnonzero(retained_mask)
rej_show = rng.choice(rejected_index, size=min(2500, len(rejected_index)), replace=False)
ret_show = rng.choice(retained_index, size=min(2500, len(retained_index)), replace=False)
ax.scatter(all_resolved.iloc[rej_show]['dhw_per_person'],
           all_resolved.iloc[rej_show]['floor_num'],
           c='grey', s=4, alpha=0.18, label='剔除样本')
ax.scatter(all_resolved.iloc[ret_show]['dhw_per_person'],
           all_resolved.iloc[ret_show]['floor_num'],
           c='darkorange', s=6, alpha=0.50, label='保留样本')
ax.set_xlabel('人均生活热水量（m3/(人·d)）')
ax.set_ylabel('楼层数')
ax.set_title('二维参数覆盖检验')
ax.legend(fontsize=9, markerscale=3)
ax.grid(alpha=0.3)

plt.tight_layout()
out_dir = PROJECT_ROOT / 'outputs_step1' / 'figures'
out_dir.mkdir(parents=True, exist_ok=True)
save_figure(fig, out_dir / 'feasibility_screening_analysis.png', dpi=300, bbox_inches='tight')
plt.show()


def js_distance_continuous(a, b, bins=40):
    """Jensen-Shannon distance on common histogram bins."""
    a = pd.Series(a).dropna().astype(float)
    b = pd.Series(b).dropna().astype(float)
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if np.isclose(lo, hi):
        return 0.0
    counts_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=False)
    counts_b, _ = np.histogram(b, bins=edges, density=False)
    pa = counts_a + 1e-12
    pb = counts_b + 1e-12
    pa = pa / pa.sum()
    pb = pb / pb.sum()
    return float(jensenshannon(pa, pb, base=2.0))


shift_rows = []
for feature in [f["name"] for f in FEATURES]:
    pre = all_resolved[feature]
    post = all_resolved.loc[retained_mask, feature]
    if pd.api.types.is_numeric_dtype(pre):
        ks_stat, ks_p = ks_2samp(pre.dropna(), post.dropna())
        js_dist = js_distance_continuous(pre, post)
        shift_rows.append({
            "feature": feature,
            "pre_mean": pre.mean(),
            "post_mean": post.mean(),
            "mean_shift_pct": (post.mean() / pre.mean() - 1) * 100 if pre.mean() != 0 else np.nan,
            "pre_std": pre.std(),
            "post_std": post.std(),
            "ks_statistic": ks_stat,
            "ks_p_value": ks_p,
            "jensen_shannon_distance": js_dist,
        })

screening_shift_df = pd.DataFrame(shift_rows).sort_values(
    ["ks_statistic", "jensen_shannon_distance"], ascending=False
)
screening_shift_df.to_csv(
    OUTPUT_DIR / "feasibility_screening_variable_distribution_shift.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=" * 60)
print("可行性筛选分析 / FEASIBILITY SCREENING ANALYSIS")
print("=" * 60)
print(f"筛选前样本数 / Pre-filter samples:  {len(raw_ratio):,}")
print(f"筛选后样本数 / Post-filter samples: {len(filtered_ratio):,}")
print(f"剔除率 / Rejection rate: {(1 - retention/100) * 100:.1f}%")
print(f"筛选前面积比: mean={raw_ratio.mean():.3f}, std={raw_ratio.std():.3f}, "
      f"min={raw_ratio.min():.3f}, max={raw_ratio.max():.3f}")
print(f"筛选后面积比: mean={filtered_ratio.mean():.3f}, std={filtered_ratio.std():.3f}, "
      f"min={filtered_ratio.min():.3f}, max={filtered_ratio.max():.3f}")
ks_stat, ks_p = ks_2samp(raw_ratio, filtered_ratio)
print(f"面积比两样本 KS 检验: D={ks_stat:.4f}, p={ks_p:.4g}")
print("\n变量级分布偏移最大的前 10 项：")
display(screening_shift_df.head(10).round(4))
print(f"\n变量分布偏移表已保存: {OUTPUT_DIR / 'feasibility_screening_variable_distribution_shift.csv'}")


<!-- CODEx 2026-06-16 reviewer-strengthening: 可行性筛选边界敏感性分析 -->
### 可行性筛选边界敏感性分析

该分析检验 0.55–0.95 可用面积比边界是否会主导后续 SRC 变量排序。其目的不是重复说明规范依据，而是量化“边界轻微浮动时核心结论是否保持稳定”。

In [ ]:

# ============================================================
# 2026-06-16 V1: feasibility-boundary sensitivity analysis
# Tests whether SRC conclusions depend on the 0.55-0.95 screening interval.
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

boundary_scenarios = [
    ("Baseline [0.55, 0.95]", 0.55, 0.95),
    ("Lower relaxed [0.50, 0.95]", 0.50, 0.95),
    ("Lower strict [0.60, 0.95]", 0.60, 0.95),
    ("Upper relaxed [0.55, 0.98]", 0.55, 0.98),
    ("Upper strict [0.55, 0.90]", 0.55, 0.90),
    ("Wide [0.50, 0.98]", 0.50, 0.98),
    ("Narrow [0.60, 0.90]", 0.60, 0.90),
]

def _src_rank_for_boundary(df_in, feature_cols, target_col):
    d = df_in[feature_cols + [target_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < max(12, min(30, len(feature_cols) + 2)):
        return pd.DataFrame(columns=["feature", "SRC", "abs_SRC", "rank_abs"])
    Xb = StandardScaler().fit_transform(d[feature_cols])
    yb = StandardScaler().fit_transform(d[[target_col]]).ravel()
    model = LinearRegression().fit(Xb, yb)
    out = pd.DataFrame({"feature": feature_cols, "SRC": model.coef_})
    out["abs_SRC"] = out["SRC"].abs()
    out["rank_abs"] = out["abs_SRC"].rank(method="first", ascending=False).astype(int)
    return out.sort_values("rank_abs")

dataset_path = DATA_DIR / "step1_simulation_dataset.csv"
if not dataset_path.exists():
    print("未发现 step1_simulation_dataset.csv；边界敏感性分析将在完成 Step 1 仿真后自动运行。")
else:
    sim_df = pd.read_csv(dataset_path)
    if "window_type_id" in sim_df.columns:
        sim_df = pd.get_dummies(sim_df, columns=["window_type_id"], prefix="window_type", drop_first=True)
    if "orientation_deg" in sim_df.columns:
        sim_df["orientation_sin"] = np.sin(np.deg2rad(sim_df["orientation_deg"]))
        sim_df["orientation_cos"] = np.cos(np.deg2rad(sim_df["orientation_deg"]))
    if "footprint_area_m2" not in sim_df.columns:
        sim_df["footprint_area_m2"] = sim_df["building_length"] * sim_df["building_width"]
    if "aspect_ratio" not in sim_df.columns:
        sim_df["aspect_ratio"] = sim_df["building_length"] / sim_df["building_width"].replace(0, np.nan)

    boundary_features = [
        'insul_thick', 'wwr', 'wall_thick',
        'u_win_n', 'u_win_s', 'u_win_e', 'u_win_w',
        'u_wall', 'u_roof', 'u_ground',
        'shgc_n', 'shgc_s', 'shgc_e', 'shgc_w',
        'roof_insul_thick', 'floor_num', 'footprint_area_m2',
        'aspect_ratio', 'floor_height', 'orientation_sin', 'orientation_cos',
        'public_area', 'room_area', 'room_count',
        'equip_power', 'dhw_per_person', 'occupancy_density', 'light_power',
        'cool_set', 'heat_set', 'dhw_temp',
        'cop_cooling', 'cop_heating', 'boiler_eff', 'fan_eff',
        'fresh_air_ach', 'operation_hours',
        'window_type_2', 'window_type_3'
    ]
    boundary_features = [c for c in boundary_features if c in sim_df.columns]
    baseline_subset = sim_df[sim_df["usable_area_ratio"].between(0.55, 0.95)].copy()
    baseline_rank = _src_rank_for_boundary(baseline_subset, boundary_features, "eui_kwh_m2")
    baseline_top10 = set(baseline_rank.head(10)["feature"])
    baseline_top18 = set(baseline_rank.head(18)["feature"])

    rows = []
    for label, lo, hi in boundary_scenarios:
        design_mask = samples_all["usable_area_ratio"].between(lo, hi)
        available = sim_df[sim_df["usable_area_ratio"].between(lo, hi)].copy()
        rank_df = _src_rank_for_boundary(available, boundary_features, "eui_kwh_m2")
        if baseline_rank.empty or rank_df.empty:
            rho = np.nan
            top10_overlap = np.nan
            top18_overlap = np.nan
        else:
            merged = baseline_rank[["feature", "rank_abs"]].merge(
                rank_df[["feature", "rank_abs"]], on="feature", suffixes=("_baseline", "_scenario")
            )
            rho = spearmanr(merged["rank_abs_baseline"], merged["rank_abs_scenario"]).correlation
            top10_overlap = len(baseline_top10 & set(rank_df.head(10)["feature"])) / max(1, len(baseline_top10))
            top18_overlap = len(baseline_top18 & set(rank_df.head(18)["feature"])) / max(1, len(baseline_top18))
        rows.append({
            "scenario": label,
            "lower_bound": lo,
            "upper_bound": hi,
            "design_space_retained_n": int(design_mask.sum()),
            "design_space_retention_pct": float(design_mask.mean() * 100),
            "available_simulated_n": int(len(available)),
            "spearman_rank_vs_baseline": rho,
            "top10_overlap_vs_baseline": top10_overlap,
            "top18_overlap_vs_baseline": top18_overlap,
        })

    boundary_df = pd.DataFrame(rows)
    boundary_df.to_csv(OUTPUT_DIR / "boundary_sensitivity_results.csv", index=False, encoding="utf-8-sig")

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), dpi=150, constrained_layout=True)
    labels = boundary_df["scenario"].str.replace(" ", "\n", regex=False)
    axes[0].barh(labels, boundary_df["design_space_retention_pct"], color="#4C78A8")
    axes[0].set_xlabel("设计空间保留率（%）")
    axes[0].set_title("可行性边界对设计空间的影响")
    axes[0].axvline(retention, color="#F58518", linestyle="--", linewidth=1.1, label="基准保留率")
    axes[0].legend(frameon=False, fontsize=8)

    x = np.arange(len(boundary_df))
    axes[1].plot(x, boundary_df["spearman_rank_vs_baseline"], marker="o", label="SRC 排名 Spearman rho", color="#4C78A8")
    axes[1].plot(x, boundary_df["top10_overlap_vs_baseline"], marker="s", label="Top-10 重叠率", color="#F58518")
    axes[1].plot(x, boundary_df["top18_overlap_vs_baseline"], marker="^", label="Top-18 重叠率", color="#54A24B")
    axes[1].set_ylim(0, 1.05)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=0, fontsize=8)
    axes[1].set_ylabel("相对于基准边界的一致性")
    axes[1].set_title("关键变量结论对筛选边界的稳健性")
    axes[1].legend(frameon=False, fontsize=8)
    save_figure(fig, FIG_DIR / "boundary_sensitivity_analysis.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("边界敏感性分析已保存：", OUTPUT_DIR / "boundary_sensitivity_results.csv")
    display(boundary_df.round(4))


<!-- CODEx bilingual cell explanation: start -->
### Cell 06 — 论文级工程示意图与研究链路图 / Publication engineering and workflow schematics

**中文说明**：调用 `tools/generate_publication_figures_v4_english.py` 中的绘图函数，生成 v4 全英文酒店热工程示意图和研究链路图。酒店图采用更接近参考图的三维轴测科研示意风格，将参数化体量、标准层平面、窗墙比传热边界、HVAC/生活热水和 EUI-OCEI 核算边界放在同一张多面板图中；研究链路图使用直线和折线展示从参数空间、LHS、EnergyPlus、SRC/SHAP、机器学习到 OCEI 敏感性分析的数据依赖关系。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。图件脚本只依赖 `matplotlib` 和项目目录结构，不依赖外部专有绘图软件。

**主要输出**：导出 `hotel_thermal_engineering_schematic_v4_en` 和 `research_pipeline_workflow_v4_en` 的 PNG、SVG、PDF 三种格式，分别用于快速预览、矢量编辑和论文/补充材料归档。

**复现提示**：运行前确认 `outputs_step1/figures/` 可写；运行后同时检查 PNG 预览和 SVG/PDF 矢量文件，重点核对全英文标签、标签间距、图例位置、箭头方向和是否存在图文遮挡。

**English explanation**: Calls the plotting functions in `tools/generate_publication_figures_v4_english.py` to create the v4 all-English hotel thermal-engineering schematic and research workflow diagram. The hotel figure uses an isometric scientific-engineering style and combines parameterised massing, typical floor zoning, window-to-wall and heat-transfer boundaries, HVAC/domestic-hot-water processing, and the EUI-OCEI accounting chain. The workflow diagram uses straight and orthogonal connectors to show dependencies from the parameter space and LHS sampling to EnergyPlus, SRC/SHAP interpretation, machine learning, and OCEI sensitivity analysis.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow and applies the shared publication plotting style. The plotting script depends only on `matplotlib` and the project directory structure, not on proprietary drawing software.

**Main outputs**: Exports `hotel_thermal_engineering_schematic_v4_en` and `research_pipeline_workflow_v4_en` in PNG, SVG, and PDF formats for preview, vector editing, and manuscript or supplementary-material archiving.

**Reproducibility note**: Before running, confirm that `outputs_step1/figures/` is writable. After running, inspect both PNG previews and SVG/PDF vector files for English-only labels, label spacing, legend placement, arrow direction, and absence of text-image overlap.
<!-- CODEx bilingual cell explanation: end -->

In [ ]:
# ---------- 3) Publication-grade English engineering and research-workflow schematics ----------
from tools.generate_publication_figures_v4_english import (
    generate_hotel_engineering_schematic_v4,
    generate_research_workflow_v4,
)

schematic_outputs = generate_hotel_engineering_schematic_v4(PAPER_ASSETS_FIG_DIR)
workflow_outputs = generate_research_workflow_v4(PAPER_ASSETS_FIG_DIR)

print("V4 English hotel thermal-engineering schematic")
for fmt, path in schematic_outputs.items():
    print(f"  {fmt}: {path}")

print("\nV4 English research workflow diagram")
for fmt, path in workflow_outputs.items():
    print(f"  {fmt}: {path}")

# Restore Chinese plotting fonts after schematic generation changes rcParams.
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


### 可行性筛选的合理性论证 (Feasibility Screening Justification)

**可行性筛选的研究依据与偏差诊断：**

0.55–0.95 的可用面积/总面积比筛选边界基于以下依据：

1. **下界（0.55）**：依据 GB 50189-2015《公共建筑节能设计标准》，酒店建筑需为客房、走道及服务空间分配充足的使用面积。若该比值低于 0.55，意味着超过 45% 的总建筑面积被结构构件、管井或未分配空间占据——这对 6–20 层的中高层酒店而言在建筑学上是不现实的。

2. **上界（0.95）**：若比值高于 0.95，则墙体、柱、电梯井、楼梯间和机电用房的总占比不足 5%——这对中高层建筑在结构上不可能实现。常规实践中，这些核心要素通常占总建筑面积的 10–30%。

3. **文献对比**：Zhang et al. (2024, Buildings 14:356) 和 Permana et al. (2023, Buildings 13:1022) 在类似酒店参数化研究中采用了可比的几何可行性检查，区间分别为 0.50–0.90 和 0.60–0.95。

4. **参数空间覆盖检验**：上图 Panel 3 展示筛选前后在关键参数二维空间中的覆盖情况，表明筛选并未系统性排除参数空间的特定区域。

77% 的剔除率反映了 LHS 在宽阔参数范围内独立采样 6 个几何变量的固有特征——许多随机组合不可避免地产生几何上不可行的建筑。这是高维建筑参数空间中拒绝采样的已知特性，**不表示选择偏差**，而是表明可行性筛选正在按预期排除物理上不可实现的方案。

**参考文献补充：**
- GB 50189-2015. 公共建筑节能设计标准 [S]. 北京: 中国建筑工业出版社, 2015.
- Zhang, J.; Yuan, C.; Yang, J.; Zhao, L. Research on Energy Consumption Prediction Models for High-Rise Hotels in Guangzhou. *Buildings* 2024, 14, 356.
- Permana, I.; Wang, F.; Agharid, A.P.; Rakshit, D.; Luo, J. Energy Consumption Analysis Using Weighted Energy Index and Energy Modeling for a Hotel Building. *Buildings* 2023, 13, 1022.


### EnergyPlus 模型可复现性文档 (Model Reproducibility Documentation)

**EnergyPlus 原型模型可复现性说明：**

**基准几何模板：**
- 酒店建筑建模为单区矩形棱柱体。建筑占地面积 = building_length × building_width（其中 building_width = building_length / aspect_ratio）。
- 总建筑高度 = floor_num × floor_height。
- 单一空调热区代表整个酒店的聚合热行为。

**暖通空调（HVAC）系统：**
- 模型使用 `HVACTemplate:Zone:IdealLoadsAirSystem`，以理想负荷方式计算供暖和供冷能量。
- 选择该系统的原因：(a) 研究重点在于设计参数对 EUI 的敏感性，而非详细 HVAC 动态特性；(b) 理想负荷为成千上万个设计方案的比较提供一致的基准线；(c) 后续工程后处理通过基于 COP 的效率核算将理想负荷转换为现场能源消耗。

**生活热水（DHW）系统：**
- 生活热水能耗在 EnergyPlus 外部通过分析计算（非 EnergyPlus 内部模拟）：DHW_energy = dhw_per_person × estimated_people × ρ × cp × ΔT × 365 / boiler_efficiency。
- 这反映了酒店 DHW 需求的稳态特性。

**人员与运行时间表：**
- 人员数量 = min(room_count × 1.6, gross_floor_area × occupancy_density)。
- 逐时运行比例基于 daily_operation_hours / 24 的简化值，裁剪至 [0.15, 1.0]，代表酒店持续运行、夜间活动减少的典型模式。

**气象文件：**
- Beijing.epw — 北京典型气象年（TMY）数据，来源于 EnergyPlus 气象数据库 (https://energyplus.net/weather)，属于中国标准气象数据（CSWD）集。

**围护结构构造：**
- 所有构造使用 `Material:NoMass`，R 值由采样的 U 值计算（R = 1/U − 表面热阻）。窗户使用 `WindowMaterial:SimpleGlazingSystem`，每个朝向独立设置 U 值和 SHGC。

**窗户构造类型定义：**
- Type 1（基准参考）：双层透明玻璃，U ≈ 1.8 W/(m²·K)，SHGC ≈ 0.55，VT ≈ 0.65
- Type 2（中等性能）：双层 Low-E 玻璃，U ≈ 1.4 W/(m²·K)，SHGC ≈ 0.40，VT ≈ 0.55
- Type 3（高性能）：三层 Low-E 玻璃，U ≈ 0.8 W/(m²·K)，SHGC ≈ 0.25，VT ≈ 0.45
- 参数表中 window_type_id 为 1、2、3 分别对应上述三类。


<!-- CODEx bilingual cell explanation: start -->
### Cell 06 — IDF 文本辅助函数 / IDF text helper functions

**中文说明**：定义 IDF 对象格式化、数值格式化、窗洞顶点生成、顶点字段展开和日运行比例时间表生成函数。这些纯函数把参数表转换为 EnergyPlus 可读对象，是批量生成 IDF 的基础。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Defines pure helper functions for IDF object formatting, numeric formatting, window-vertex generation, vertex-field expansion, and compact daily fraction schedules. These functions translate parameter-table rows into EnergyPlus-readable objects and form the basis of batch IDF generation.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 3) IDF-generation helper functions ----------

def idf_object(obj_name: str, fields: List[str]) -> str:
    lines = [f"{obj_name},"]
    for i, field in enumerate(fields):
        end = ";" if i == len(fields) - 1 else ","
        lines.append(f"  {field}{end}")
    return "\n".join(lines) + "\n\n"


def fmt(x) -> str:
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)):
        return f"{float(x):.6f}".rstrip("0").rstrip(".")
    return str(x)


def make_window_vertices(length: float, width: float, height: float, wall: str, wwr: float):
    sill = 1.0
    clear_h = max(0.8, height - 1.6)

    if wall in ["South", "North"]:
        wall_w = length
    else:
        wall_w = width

    target_area = wall_w * height * wwr
    win_w = wall_w * 0.80
    win_h = min(clear_h, max(0.6, target_area / max(win_w, 0.1)))
    win_w = min(wall_w * 0.90, max(0.8, target_area / max(win_h, 0.1)))

    x0 = (length - win_w) / 2.0
    y0 = (width - win_w) / 2.0
    z1 = sill
    z2 = sill + win_h

    if wall == "South":
        return [(x0, 0, z2), (x0, 0, z1), (x0 + win_w, 0, z1), (x0 + win_w, 0, z2)]

    if wall == "North":
        return [(x0 + win_w, width, z2), (x0 + win_w, width, z1), (x0, width, z1), (x0, width, z2)]

    if wall == "East":
        return [(length, y0, z2), (length, y0, z1), (length, y0 + win_w, z1), (length, y0 + win_w, z2)]

    if wall == "West":
        return [(0, y0 + win_w, z2), (0, y0 + win_w, z1), (0, y0, z1), (0, y0, z2)]


def vertices_to_fields(vertices: List[Tuple[float, float, float]]) -> List[str]:
    fields = [str(len(vertices))]
    for x, y, z in vertices:
        fields.extend([fmt(x), fmt(y), fmt(z)])
    return fields


def make_daily_fraction_schedule(name: str, on_fraction: float) -> str:
    hours_on = on_fraction * 24.0
    start = max(0, 12.0 - hours_on / 2.0)
    end = min(24, start + hours_on)
    s1 = int(math.floor(start))
    s2 = int(math.ceil(end))
    return idf_object("Schedule:Compact", [
        name,
        "Fraction",
        "Through: 12/31",
        "For: AllDays",
        f"Until: {s1:02d}:00, 0.15",
        f"Until: {s2:02d}:00, 1.0",
        "Until: 24:00, 0.15",
    ])


<!-- CODEx bilingual cell explanation: start -->
### Cell 07 — 酒店原型 IDF 生成 / Hotel prototype IDF generation

**中文说明**：根据单行参数构建简化酒店 EnergyPlus IDF：包括单区矩形体量、四向窗、Material:NoMass 围护结构、人员/照明/设备内扰、渗透、新风、恒温器、IdealLoadsAirSystem 和年度输出变量。该 cell 只生成仿真输入文件，不直接运行 EnergyPlus。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Builds a simplified hotel EnergyPlus IDF from each parameter row, including the single-zone rectangular massing, four-orientation windows, Material:NoMass envelope, people/lights/equipment internal gains, infiltration, outdoor air, thermostat schedules, IdealLoadsAirSystem, and annual output variables. This cell generates simulation input files only; it does not launch EnergyPlus.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 4) Generate simplified hotel IDFs with pure functions ----------

import shutil

if CONFIG.get("clean_idf_dir", False) and IDF_DIR.exists():
    shutil.rmtree(IDF_DIR)
if CONFIG.get("clean_run_dir", False) and RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

IDF_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


def build_hotel_idf_text(row: pd.Series) -> str:
    sid = row["sample_id"]
    L = float(row["building_length"])
    W = float(row["building_width"])
    H = float(row["building_height_m"])

    south_wall = [(0, 0, H), (0, 0, 0), (L, 0, 0), (L, 0, H)]
    east_wall  = [(L, 0, H), (L, 0, 0), (L, W, 0), (L, W, H)]
    north_wall = [(L, W, H), (L, W, 0), (0, W, 0), (0, W, H)]
    west_wall  = [(0, W, H), (0, W, 0), (0, 0, 0), (0, 0, H)]

    floor = [(0, W, 0), (L, W, 0), (L, 0, 0), (0, 0, 0)]
    roof  = [(0, 0, H), (L, 0, H), (L, W, H), (0, W, H)]

    south_win = make_window_vertices(L, W, H, "South", float(row["wwr"]))
    east_win  = make_window_vertices(L, W, H, "East",  float(row["wwr"]))
    north_win = make_window_vertices(L, W, H, "North", float(row["wwr"]))
    west_win  = make_window_vertices(L, W, H, "West",  float(row["wwr"]))

    idf = []
    idf.append("! ------------------------------------------------------------\n")
    idf.append(f"! Auto-generated hotel baseline model: {sid}\n")
    idf.append("! Route: LHS parameters -> function-generated IDF -> EnergyPlus loads -> engineering end-use postprocess\n")
    idf.append("! ------------------------------------------------------------\n\n")

    idf.append(idf_object("Version", ["25.2"]))
    idf.append(idf_object("Timestep", ["6"]))
    idf.append(idf_object("Building", [
        sid,
        fmt(row["orientation_deg"]),
        "Suburbs",
        "0.04",
        "0.4",
        "FullExterior",
        "25",
        "6",
    ]))
    idf.append(idf_object("SimulationControl", ["No", "No", "No", "NO", "Yes"]))
    idf.append(idf_object("RunPeriod", [
        "Annual",   # Name
        "1",        # Begin Month
        "1",        # Begin Day of Month
        "",         # Begin Year
        "12",       # End Month
        "31",       # End Day of Month
        "",         # End Year
        "",         # Day of Week for Start Day
        "Yes",      # Use Weather File Holidays and Special Days
        "Yes",      # Use Weather File Daylight Saving Period
        "No",       # Apply Weekend Holiday Rule
        "Yes",      # Use Weather File Rain Indicators
        "Yes",      # Use Weather File Snow Indicators
        "No",       # Treat Weather as Actual
    ]))
    idf.append(idf_object("GlobalGeometryRules", ["UpperLeftCorner", "CounterClockWise", "Relative"]))
    idf.append(idf_object("Site:GroundTemperature:BuildingSurface", ["18", "18", "18", "18", "18", "18", "18", "18", "18", "18", "18", "18"]))

    # schedule limits
    idf.append(idf_object("ScheduleTypeLimits", ["Fraction", "0", "1", "Continuous"]))
    idf.append(idf_object("ScheduleTypeLimits", ["Temperature", "-60", "200", "Continuous", "Temperature"]))
    idf.append(idf_object("ScheduleTypeLimits", ["Any Number"]))
    idf.append(idf_object("Schedule:Constant", ["Always On", "Fraction", "1.0"]))
    idf.append(idf_object("Schedule:Constant", ["Heat Setpoint Sch", "Temperature", fmt(row["heat_set"])]))
    idf.append(idf_object("Schedule:Constant", ["Cool Setpoint Sch", "Temperature", fmt(row["cool_set"])]))
    idf.append(idf_object("Schedule:Constant", ["Activity Level Sch", "Any Number", "120"]))
    idf.append(make_daily_fraction_schedule("OccFracSch", float(row["schedule_on_fraction"])))
    idf.append(make_daily_fraction_schedule("LoadFracSch", float(row["schedule_on_fraction"])))

    # envelope
    idf.append(idf_object("Material:NoMass", ["Wall_Res", "Rough", fmt(row["wall_R_m2K_W"]), "0.90", "0.70", "0.70"]))
    idf.append(idf_object("Material:NoMass", ["Roof_Res", "Rough", fmt(row["roof_R_m2K_W"]), "0.90", "0.70", "0.70"]))
    idf.append(idf_object("Material:NoMass", ["Floor_Res", "Rough", fmt(row["floor_R_m2K_W"]), "0.90", "0.70", "0.70"]))
    idf.append(idf_object("Construction", ["ExtWallConstr", "Wall_Res"]))
    idf.append(idf_object("Construction", ["RoofConstr", "Roof_Res"]))
    idf.append(idf_object("Construction", ["FloorConstr", "Floor_Res"]))

    for ori in ["N", "S", "E", "W"]:
        idf.append(idf_object("WindowMaterial:SimpleGlazingSystem", [
            f"GLZ_{ori}",
            fmt(row[f"u_win_{ori.lower()}"]),
            fmt(row[f"shgc_{ori.lower()}"]),
            fmt(row["glass_vt"]),
        ]))
        idf.append(idf_object("Construction", [f"WINCON_{ori}", f"GLZ_{ori}"]))

    idf.append(idf_object("Zone", ["HotelZone"]))

    # surfaces
    idf.append(idf_object("BuildingSurface:Detailed", [
        "SouthWall", "Wall", "ExtWallConstr", "HotelZone", "", "Outdoors", "", "SunExposed", "WindExposed", "0.5", *vertices_to_fields(south_wall)
    ]))
    idf.append(idf_object("BuildingSurface:Detailed", [
        "EastWall", "Wall", "ExtWallConstr", "HotelZone", "", "Outdoors", "", "SunExposed", "WindExposed", "0.5", *vertices_to_fields(east_wall)
    ]))
    idf.append(idf_object("BuildingSurface:Detailed", [
        "NorthWall", "Wall", "ExtWallConstr", "HotelZone", "", "Outdoors", "", "SunExposed", "WindExposed", "0.5", *vertices_to_fields(north_wall)
    ]))
    idf.append(idf_object("BuildingSurface:Detailed", [
        "WestWall", "Wall", "ExtWallConstr", "HotelZone", "", "Outdoors", "", "SunExposed", "WindExposed", "0.5", *vertices_to_fields(west_wall)
    ]))
    idf.append(idf_object("BuildingSurface:Detailed", [
        "GroundFloor", "Floor", "FloorConstr", "HotelZone", "", "Ground", "", "NoSun", "NoWind", "1.0", *vertices_to_fields(floor)
    ]))
    idf.append(idf_object("BuildingSurface:Detailed", [
        "Roof", "Roof", "RoofConstr", "HotelZone", "", "Outdoors", "", "SunExposed", "WindExposed", "0.0", *vertices_to_fields(roof)
    ]))

    idf.append(idf_object("FenestrationSurface:Detailed", [
        "SouthWin", "Window", "WINCON_S", "SouthWall", "", "", "", "1.0", *vertices_to_fields(south_win)
    ]))
    idf.append(idf_object("FenestrationSurface:Detailed", [
        "EastWin", "Window", "WINCON_E", "EastWall", "", "", "", "1.0", *vertices_to_fields(east_win)
    ]))
    idf.append(idf_object("FenestrationSurface:Detailed", [
        "NorthWin", "Window", "WINCON_N", "NorthWall", "", "", "", "1.0", *vertices_to_fields(north_win)
    ]))
    idf.append(idf_object("FenestrationSurface:Detailed", [
        "WestWin", "Window", "WINCON_W", "WestWall", "", "", "", "1.0", *vertices_to_fields(west_win)
    ]))

    # internal gains
    idf.append(idf_object("People", [
        "HotelPeople", "HotelZone", "OccFracSch", "People/Area", "", fmt(row["occupancy_density"]), "", "0.30", "autocalculate", "Activity Level Sch"
    ]))
    idf.append(idf_object("Lights", [
        "HotelLights", "HotelZone", "LoadFracSch", "Watts/Area", "", fmt(row["light_power"]), "", "0.0", "0.42", "0.18", "1.0", "GeneralLights"
    ]))
    idf.append(idf_object("ElectricEquipment", [
        "HotelEquip", "HotelZone", "LoadFracSch", "Watts/Area", "", fmt(row["equip_power"]), "", "0.0", "0.30", "0.0", "PlugLoads"
    ]))
    idf.append(idf_object("ZoneInfiltration:DesignFlowRate", [
        "HotelInfiltration", "HotelZone", "Always On", "AirChanges/Hour", "", "", "", fmt(max(0.1, row["fresh_air_ach"] * 0.30))
    ]))

    # thermostat + ideal loads
    idf.append(idf_object("HVACTemplate:Thermostat", [
        "Hotel Thermostat", "Heat Setpoint Sch", "", "Cool Setpoint Sch", ""
    ]))
    idf.append(idf_object("HVACTemplate:Zone:IdealLoadsAirSystem", [
        "HotelZone",
        "Hotel Thermostat",
        "Always On",
        "50",
        "13",
        "0.0156",
        "0.0077",
        "NoLimit",
        "",
        "",
        "NoLimit",
        "",
        "",
        "",
        "",
        "ConstantSensibleHeatRatio",
        "0.7",
        "60",
        "None",
        "30",
        "Flow/Area",
        "0.0",
        fmt(row["fresh_air_ach"] / 3600.0 * row["floor_height"]),
        "0.0",
        "",
        "None",
        "NoEconomizer",
        "None",
        "0.70",
        "0.65",
    ]))

    # reporting
    idf.append(idf_object("OutputControl:Table:Style", ["CommaAndHTML"]))
    idf.append(idf_object("Output:SQLite", ["SimpleAndTabular"]))
    idf.append(idf_object("Output:VariableDictionary", ["Regular"]))
    idf.append(idf_object("Output:Table:SummaryReports", ["AnnualBuildingUtilityPerformanceSummary"]))
    # idf.append(idf_object("Output:Table:SummaryReports", ["EndUseEnergyConsumptionSummary"]))
    idf.append(idf_object("Output:Variable", ["*", "Zone Ideal Loads Supply Air Total Heating Energy", "RunPeriod"]))
    idf.append(idf_object("Output:Variable", ["*", "Zone Ideal Loads Supply Air Total Cooling Energy", "RunPeriod"]))
    idf.append(idf_object("Output:Variable", ["*", "Zone Ideal Loads Zone Total Heating Energy", "RunPeriod"]))
    idf.append(idf_object("Output:Variable", ["*", "Zone Ideal Loads Zone Total Cooling Energy", "RunPeriod"]))

    return "".join(idf)


def generate_idf_files(samples_df: pd.DataFrame, out_dir: Path) -> List[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    files = []
    for _, row in samples_df.iterrows():
        path = out_dir / f"{row['sample_id']}.idf"
        path.write_text(build_hotel_idf_text(row), encoding="utf-8")
        files.append(path)
    return files

idf_files = generate_idf_files(samples, IDF_DIR)
print(f"Generated IDF count: {len(idf_files)}")
print(idf_files[:3])


<!-- CODEx bilingual cell explanation: start -->
### Cell 08 — 单样本 EnergyPlus 调用 / Single-sample EnergyPlus execution

**中文说明**：把 IDF、Energy+.ini 和 Energy+.idd 复制到样本运行目录，先执行 ExpandObjects，再调用 EnergyPlus，并记录 stdout/stderr、错误文件、严重错误和运行状态。该函数为大批量仿真提供统一的失败诊断格式。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Copies the IDF, Energy+.ini, and Energy+.idd into the sample run directory, executes ExpandObjects, launches EnergyPlus, and records stdout/stderr tails, error-file content flags, severe/fatal errors, and success status. The function provides a uniform failure-diagnostic record for large simulation batches.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
import shutil

def run_single_simulation(
    idf_path: Path,
    weather_path: Path,
    energyplus_exe: Path,
    run_root: Path,
    timeout_seconds: int = 900,
    expandobjects_exe: Path = None,
) -> Dict:
    sample_name = idf_path.stem
    sample_run_dir = run_root / sample_name
    sample_run_dir.mkdir(parents=True, exist_ok=True)

    # Standard input filenames inside each run directory.
    local_idf = sample_run_dir / "in.idf"
    shutil.copy2(idf_path, local_idf)

    # Copy Energy+.ini and Energy+.idd to prevent ExpandObjects from failing due to missing files.
    ep_root = energyplus_exe.parent
    ini_src = ep_root / "Energy+.ini"
    idd_src = ep_root / "Energy+.idd"

    if ini_src.exists():
        shutil.copy2(ini_src, sample_run_dir / "Energy+.ini")
    if idd_src.exists():
        shutil.copy2(idd_src, sample_run_dir / "Energy+.idd")

    expanded_idf = local_idf
    expand_stdout = ""
    expand_stderr = ""

    if expandobjects_exe is not None and Path(expandobjects_exe).exists():
        exp_completed = subprocess.run(
            [str(expandobjects_exe)],
            cwd=sample_run_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=timeout_seconds,
        )
        expand_stdout = exp_completed.stdout[-1500:]
        expand_stderr = exp_completed.stderr[-1500:]

        candidate = sample_run_dir / "expanded.idf"
        if candidate.exists():
            expanded_idf = candidate
        else:
            return {
                "sample_id": sample_name,
                "returncode": exp_completed.returncode,
                "stdout_tail": expand_stdout,
                "stderr_tail": expand_stderr,
                "run_dir": str(sample_run_dir),
                "has_severe_error": True,
                "success": False,
            }

    cmd = [
        str(energyplus_exe),
        "-w", str(weather_path),
        "-d", str(sample_run_dir),
        str(expanded_idf),
    ]

    completed = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=timeout_seconds,
    )

    err_text = ""
    err_file = sample_run_dir / "eplusout.err"
    if err_file.exists():
        err_text = err_file.read_text(encoding="utf-8", errors="ignore")

    has_severe = ("** Severe  **" in err_text) or ("**  Fatal  **" in err_text)

    return {
        "sample_id": sample_name,
        "returncode": completed.returncode,
        "stdout_tail": (expand_stdout + "\n" + completed.stdout)[-1500:],
        "stderr_tail": (expand_stderr + "\n" + completed.stderr)[-1500:],
        "run_dir": str(sample_run_dir),
        "has_severe_error": has_severe,
        "success": (completed.returncode == 0) and (not has_severe),
    }

<!-- CODEx bilingual cell explanation: start -->
### Cell 09 — SQLite 结果解析与工程后处理 / SQLite parsing and engineering post-processing

**中文说明**：从 EnergyPlus SQLite 文件读取建筑面积、理想供冷/供热负荷，并通过照明、设备、风机、COP 换算和生活热水公式计算电力、天然气、总能耗与 EUI。该 cell 明确把模拟负荷转换为工程能耗指标的计算边界。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Reads conditioned area and ideal cooling/heating loads from the EnergyPlus SQLite output, then computes lighting, equipment, fan electricity, COP-adjusted cooling/heating electricity, domestic hot water, natural gas, total site energy, and EUI. It makes the engineering boundary from simulated loads to operational energy metrics explicit.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 6) Parse EnergyPlus outputs and apply engineering post-processing ----------
J_TO_KWH = 1 / 3.6e6


def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def parse_sqlite_result(run_dir: Path) -> Dict[str, float]:
    sql_path = run_dir / "eplusout.sql"
    if not sql_path.exists():
        return {}

    conn = sqlite3.connect(sql_path)
    out = {}

    try:
        # 1) Building area.
        q_area = """
        SELECT Value
        FROM TabularDataWithStrings
        WHERE ReportName='AnnualBuildingUtilityPerformanceSummary'
          AND TableName='Building Area'
          AND RowName='Net Conditioned Building Area'
        LIMIT 1
        """
        area_df = pd.read_sql_query(q_area, conn)
        if not area_df.empty:
            out["gross_floor_area_sql_m2"] = _safe_float(area_df.iloc[0, 0])

        # 2) Run-period variables: normalize Reporting频数 to handle RunPeriod / Run Period naming differences.
        q_var = """
        SELECT
            TRIM(d.Name) AS variable_name,
            REPLACE(LOWER(TRIM(d.Reporting频数)), ' ', '') AS freq_norm,
            SUM(r.Value) AS total_value
        FROM ReportData AS r
        JOIN ReportDataDictionary AS d
          ON r.ReportDataDictionaryIndex = d.ReportDataDictionaryIndex
        GROUP BY TRIM(d.Name), REPLACE(LOWER(TRIM(d.Reporting频数)), ' ', '')
        """
        var_df = pd.read_sql_query(q_var, conn)

        if not var_df.empty:
            runperiod_df = var_df.loc[var_df["freq_norm"] == "runperiod"].copy()
            runperiod_df["name_norm"] = runperiod_df["variable_name"].str.strip().str.lower()

            def get_kwh(exact_names, fallback_terms=None):
                # Try exact matching first.
                for name in exact_names:
                    hit = runperiod_df.loc[
                        runperiod_df["name_norm"] == name.strip().lower(),
                        "total_value"
                    ]
                    if len(hit):
                        return float(hit.iloc[0]) * J_TO_KWH

                # Fall back to fuzzy matching.
                if fallback_terms:
                    mask = runperiod_df["name_norm"].apply(
                        lambda s: all(term in s for term in fallback_terms)
                    )
                    hit = runperiod_df.loc[mask, "total_value"]
                    if len(hit):
                        return float(hit.iloc[0]) * J_TO_KWH

                return np.nan

            out["ideal_heating_supply_kwh"] = get_kwh(
                ["Zone Ideal Loads Supply Air Total Heating Energy"],
                fallback_terms=["ideal loads", "supply air", "heating", "energy"]
            )
            out["ideal_cooling_supply_kwh"] = get_kwh(
                ["Zone Ideal Loads Supply Air Total Cooling Energy"],
                fallback_terms=["ideal loads", "supply air", "cooling", "energy"]
            )
            out["ideal_heating_zone_kwh"] = get_kwh(
                ["Zone Ideal Loads Zone Total Heating Energy"],
                fallback_terms=["ideal loads", "zone total", "heating", "energy"]
            )
            out["ideal_cooling_zone_kwh"] = get_kwh(
                ["Zone Ideal Loads Zone Total Cooling Energy"],
                fallback_terms=["ideal loads", "zone total", "cooling", "energy"]
            )

    finally:
        conn.close()

    return out


def engineering_postprocess(row: pd.Series, sim: Dict[str, float]) -> Dict[str, float]:
    # Prefer supply-air total loads; fall back to zone total loads if needed.
    cooling_load_kwh = sim.get("ideal_cooling_supply_kwh", sim.get("ideal_cooling_zone_kwh", np.nan))
    heating_load_kwh = sim.get("ideal_heating_supply_kwh", sim.get("ideal_heating_zone_kwh", np.nan))

    area = float(row["gross_floor_area_m2"])
    volume = float(row["building_volume_m3"])
    op_hours = float(row["operation_hours"])
    people = float(row["estimated_people"])

    # Lighting and equipment electricity: W/m2 * area * annual operating hours.
    lights_kwh = float(row["light_power"]) * area * op_hours / 1000.0
    equip_kwh = float(row["equip_power"]) * area * op_hours / 1000.0

    # Fan electricity using a transparent simplified equation.
    oa_flow_m3s = float(row["fresh_air_ach"]) * volume / 3600.0
    fan_kw = oa_flow_m3s * CONFIG["fan_delta_p_pa"] / max(1e-6, float(row["fan_eff"])) / 1000.0
    fan_kwh = fan_kw * op_hours

    # Cooling and heating electricity are converted from ideal loads using COP values.
    cooling_elec_kwh = np.nan if pd.isna(cooling_load_kwh) else cooling_load_kwh / max(1e-6, float(row["cop_cooling"]))
    heating_elec_kwh = np.nan if pd.isna(heating_load_kwh) else heating_load_kwh / max(1e-6, float(row["cop_heating"]))

    # Domestic hot-water energy: m3/person/day -> kg/day -> kWh/a.
    dhw_m3_day = float(row["dhw_per_person"]) * people
    delta_t = max(1.0, float(row["dhw_temp"]) - CONFIG["dhw_cold_water_temp_c"])
    dhw_thermal_kwh = dhw_m3_day * CONFIG["dhw_density_kg_m3"] * CONFIG["water_cp_kj_kgk"] * delta_t / 3600.0 * 365.0
    dhw_gas_kwh = dhw_thermal_kwh / max(1e-6, float(row["boiler_eff"]))

    electricity_kwh = np.nansum([cooling_elec_kwh, heating_elec_kwh, lights_kwh, equip_kwh, fan_kwh])
    natural_gas_kwh = dhw_gas_kwh
    total_energy_kwh = electricity_kwh + natural_gas_kwh
    eui = total_energy_kwh / max(1e-6, area)

    return {
        "gross_floor_area_m2": area,
        "ideal_cooling_load_kwh": cooling_load_kwh,
        "ideal_heating_load_kwh": heating_load_kwh,
        "cooling_electricity_kwh": cooling_elec_kwh,
        "heating_electricity_kwh": heating_elec_kwh,
        "lighting_electricity_kwh": lights_kwh,
        "equipment_electricity_kwh": equip_kwh,
        "fan_electricity_kwh": fan_kwh,
        "electricity_kwh": electricity_kwh,
        "dhw_thermal_kwh": dhw_thermal_kwh,
        "natural_gas_kwh": natural_gas_kwh,
        "site_energy_kwh": total_energy_kwh,
        "eui_kwh_m2": eui,
    }

<!-- CODEx bilingual cell explanation: start -->
### Cell 10 — 批量仿真与数据集写出 / Batch simulation and dataset export

**中文说明**：在 run_energyplus 为真时逐样本运行 EnergyPlus、解析结果并写出 simulation_log.csv 与 step1_simulation_dataset.csv；在调试模式下只提示已跳过仿真。该 cell 是生成后续三个 notebook 输入数据的唯一入口。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: When run_energyplus is true, iterates through all feasible samples, runs EnergyPlus, parses results, and writes simulation_log.csv plus step1_simulation_dataset.csv. In debug mode it reports that simulations were skipped. This cell is the sole producer of the dataset consumed by the next three notebooks.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
energyplus_exe = Path(CONFIG["energyplus_exe"])
weather_path = Path(CONFIG["weather_epw"])
expandobjects_exe = Path(CONFIG["expandobjects_exe"])

if CONFIG["run_energyplus"]:
    assert energyplus_exe.exists(), f"EnergyPlus not found: {energyplus_exe}"
    assert weather_path.exists(), f"EPW file not found: {weather_path}"
    assert expandobjects_exe.exists(), f"ExpandObjects not found: {expandobjects_exe}"

    simulation_logs = []
    dataset_rows = []

    for i, (_, row) in enumerate(samples.iterrows(), start=1):
        idf_path = IDF_DIR / f"{row['sample_id']}.idf"

        log = run_single_simulation(
            idf_path=idf_path,
            weather_path=weather_path,
            energyplus_exe=energyplus_exe,
            run_root=RUN_DIR,
            timeout_seconds=CONFIG["timeout_seconds"],
            expandobjects_exe=expandobjects_exe,
        )
        simulation_logs.append(log)

        if log["success"]:
            sim = parse_sqlite_result(Path(log["run_dir"]))
            post = engineering_postprocess(row, sim)
            dataset_rows.append({**row.to_dict(), **sim, **post})

        print(f"[{i}/{len(samples)}] {row['sample_id']} success={log['success']}")

    logs_df = pd.DataFrame(simulation_logs)
    logs_df.to_csv(OUTPUT_DIR / "simulation_log.csv", index=False, encoding="utf-8-sig")

    dataset = pd.DataFrame(dataset_rows)
    dataset.to_csv(DATA_DIR / "step1_simulation_dataset.csv", index=False, encoding="utf-8-sig")

    if not dataset.empty and {"sample_id", "site_energy_kwh", "eui_kwh_m2"}.issubset(dataset.columns):
        print(dataset[["sample_id", "site_energy_kwh", "eui_kwh_m2"]].head())
    else:
        print("No simulation dataset has been generated successfully.")
        print("Please first check simulation_log.csv and the eplusout.err file in the corresponding sample directory.")
        if not logs_df.empty:
            print(logs_df[["sample_id", "success", "returncode", "has_severe_error", "run_dir"]])

else:
    print("Current CONFIG['run_energyplus'] = False; IDFs are generated but simulations are not executed.")
    print("First set the sample size to 3-5, confirm that all runs pass, and then increase it to 300.")

<!-- CODEx bilingual cell explanation: start -->
### Cell 11 — Step 1 探索性图表 / Step 1 exploratory figures

**中文说明**：读取已生成的数据集并绘制 EUI 分布、总能耗分布、平均终端能耗组成和变量相关矩阵，用于检查仿真输出是否落在合理范围内并发现异常列。图表保存到 outputs_step1/figures。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Reads the generated dataset and plots the EUI distribution, site-energy distribution, average end-use/load components, and selected-variable correlation matrix. These figures check whether simulation outputs are in a plausible range and help detect anomalous columns. The files are saved under outputs_step1/figures.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 8) Publication-style visualization ----------
dataset_path = DATA_DIR / "step1_simulation_dataset.csv"
if dataset_path.exists():
    df = pd.read_csv(dataset_path)

    fig, ax = plt.subplots(figsize=(8.8, 5.0))
    ax.hist(df["eui_kwh_m2"].dropna(), bins=28)
    ax.set_title("酒店建筑 EUI 分布")
    ax.set_xlabel("EUI（kWh/m2·a）")
    ax.set_ylabel("频数")
    fig.tight_layout()
    save_figure(fig, FIG_DIR / "eui_distribution.png", bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(8.8, 5.0))
    ax.hist(df["site_energy_kwh"].dropna(), bins=28)
    ax.set_title("酒店建筑总能耗分布")
    ax.set_xlabel("能耗（kWh）")
    ax.set_ylabel("频数")
    fig.tight_layout()
    save_figure(fig, FIG_DIR / "site_energy_distribution.png", bbox_inches="tight")
    plt.show()

    plot_cols = [
        "electricity_kwh",
        "natural_gas_kwh",
        "ideal_cooling_load_kwh",
        "ideal_heating_load_kwh",
        "lighting_electricity_kwh",
        "equipment_electricity_kwh",
        "fan_electricity_kwh",
    ]
    
    available = [c for c in plot_cols if c in df.columns]
    mean_vals = df[available].mean().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    ax.bar(mean_vals.index, mean_vals.values)
    ax.set_title("平均终端能耗/负荷组成")
    ax.set_ylabel("kWh/a")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    save_figure(fig, FIG_DIR / "average_enduses.png", bbox_inches="tight")
    plt.show()

    corr_cols = [c for c in [
        "eui_kwh_m2", "site_energy_kwh", "wwr", "u_wall", "u_roof", "floor_num", "room_count",
        "light_power", "equip_power", "fresh_air_ach", "cool_set", "heat_set", "cop_cooling", "cop_heating"
    ] if c in df.columns]
    corr = df[corr_cols].corr(numeric_only=True)

    fig, ax = plt.subplots(figsize=(8.0, 6.5))
    im = ax.imshow(corr, aspect="auto")
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    ax.set_title("选定变量相关矩阵")
    fig.colorbar(im, ax=ax, shrink=0.85)
    fig.tight_layout()
    save_figure(fig, FIG_DIR / "correlation_matrix.png", bbox_inches="tight")
    plt.show()
else:
    print("step1_simulation_dataset.csv has not been generated yet. Please run the simulations first.")

<!-- CODEx bilingual cell explanation: start -->
### Cell 12 — 模拟 EUI 与实测基准对比 / Simulated EUI versus measured benchmark

**中文说明**：把模拟 EUI 分布与北京酒店实测 EUI 文献统计和 GB/T 51161 酒店能耗基准对比，输出均值差异和百分比偏差。该分析将模型性能表述限定为代理模型保真度，并明确模拟到真实建筑的迁移差距。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Compares the simulated EUI distribution with published measured EUI statistics for Beijing hotels and GB/T 51161 hotel benchmarks, then reports the mean difference and percentage bias. This analysis reframes model performance as surrogate fidelity and makes the simulation-to-real-building transfer gap explicit.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ============================================================
# Real-building EUI benchmark comparison
# Compare simulated EUI distribution against published Beijing hotel
# measured data to address the sim-to-real transfer gap.
# ============================================================

# Published Beijing hotel EUI benchmarks (Chen, Tan & Berardi, 2018; 56 hotels)
measured_stats = {
    'source': 'Chen, Tan & Berardi (2018) — 56 Beijing hotels',
    'mean': 123.03,   # kWh/m2.a
    'min': 44.76,
    'max': 342.47,
    'std': 45.5,      # estimated from right-skewed distribution
    'q25': 88.0,      # estimated from lognormal fit
    'q50': 110.0,
    'q75': 145.0,
}

# Chinese national benchmark for cold-zone hotels (GB/T 51161-2016)
national_benchmarks = {
    'Cat A 3-star': 110,
    'Cat A 4-star': 135,
    'Cat A 5-star': 160,
    'Cat B 3-star': 160,
    'Cat B 4-star': 200,
    'Cat B 5-star': 240,
}

# Check if simulation data exists (skip if not yet generated)
dataset_path = PROJECT_ROOT / 'data' / 'step1_simulation_dataset.csv'
if not dataset_path.exists():
    print('WARNING: step1_simulation_dataset.csv not found. Simulations have not been run yet.')
    print('Skipping real-building EUI comparison. Run EnergyPlus simulations first (set CONFIG["run_energyplus"] = True).')
else:
    df_eui = pd.read_csv(dataset_path)
    sim_eui = df_eui['eui_kwh_m2'].dropna()

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

    # Simulated distribution
    ax.hist(sim_eui, bins=50, color='steelblue', edgecolor='white', alpha=0.7,
            density=True, label=f'Simulated (n={len(sim_eui):,}, mean={sim_eui.mean():.1f})')

    # Overlay measured range
    ax.axvline(measured_stats['mean'], color='darkred', linestyle='-', linewidth=2.5,
               label=f"Measured mean: {measured_stats['mean']:.0f} (56 Beijing hotels)")
    ax.axvspan(measured_stats['q25'], measured_stats['q75'], alpha=0.12, color='red',
               label=f"Measured IQR: [{measured_stats['q25']:.0f}, {measured_stats['q75']:.0f}]")

    # National benchmarks
    colors = ['darkgreen', 'seagreen', 'forestgreen', 'sienna', 'chocolate', 'firebrick']
    for (label, val), c in zip(national_benchmarks.items(), colors):
        ax.axvline(val, color=c, linestyle=':', linewidth=1, alpha=0.6)
    ax.text(245, ax.get_ylim()[1]*0.95, 'National benchmarks (GB/T 51161):', fontsize=8, ha='right')

    ax.set_xlabel('建筑能源使用强度 EUI（kWh/(m2·a)）', fontsize=13)
    ax.set_ylabel('Probability 密度', fontsize=13)
    ax.set_title('模拟 EUI 与北京酒店实测数据对比', fontsize=14)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

    out_dir = PROJECT_ROOT / 'outputs_step1' / 'figures'
    out_dir.mkdir(parents=True, exist_ok=True)
    save_figure(fig, out_dir / 'eui_measured_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Quantitative comparison
    print("=" * 50)
    print("SIM-TO-REAL EUI COMPARISON")
    print("=" * 50)
    print(f"Simulated EUI:  mean={sim_eui.mean():.1f}, median={sim_eui.median():.1f}, "
          f"std={sim_eui.std():.1f}, range=[{sim_eui.min():.1f}, {sim_eui.max():.1f}]")
    print(f"Measured EUI:   mean={measured_stats['mean']:.1f}, "
          f"range=[{measured_stats['min']:.1f}, {measured_stats['max']:.1f}]")
    print(f"Mean difference: {sim_eui.mean() - measured_stats['mean']:.1f} kWh/m2.a "
          f"({(sim_eui.mean()/measured_stats['mean'] - 1)*100:.1f}%)")
    print(f"NOTE: Simulated mean is {(sim_eui.mean()/measured_stats['mean'] - 1)*100:.1f}% higher "
          f"than measured mean — see Discussion for sim-to-real gap analysis.")


<!-- CODEx 2026-06-16 reviewer-strengthening: 模拟到真实建筑偏差的分层分解 -->
### 模拟到真实建筑偏差的分层分解

该分析将总体模拟偏差拆分到高度与占地尺度分层中，用于说明模型适用边界：代理模型主要服务概念设计阶段的方案比选，而不是替代真实建筑校准模型。

In [ ]:

# ============================================================
# 2026-06-16 V1: stratified sim-to-real bias decomposition
# Explains where the simulated-versus-measured EUI gap is largest.
# ============================================================

dataset_path = DATA_DIR / "step1_simulation_dataset.csv"
if not dataset_path.exists():
    print("未发现 step1_simulation_dataset.csv；偏差分层分析将在完成 Step 1 仿真后自动运行。")
else:
    df_bias = pd.read_csv(dataset_path).copy()
    measured_mean = measured_stats["mean"]

    df_bias["height_stratum"] = pd.cut(
        df_bias["floor_num"],
        bins=[-np.inf, 8, 15, np.inf],
        labels=["低层（≤8层）", "中高层（9–15层）", "高层（>15层）"]
    )
    df_bias["footprint_stratum"] = pd.qcut(
        df_bias["footprint_area_m2"].rank(method="first"),
        q=3,
        labels=["小占地", "中占地", "大占地"]
    )

    bias_rows = []
    for h in df_bias["height_stratum"].cat.categories:
        for f in df_bias["footprint_stratum"].cat.categories:
            sub = df_bias[(df_bias["height_stratum"] == h) & (df_bias["footprint_stratum"] == f)]
            if sub.empty:
                continue
            sim_mean = sub["eui_kwh_m2"].mean()
            bias_rows.append({
                "height_stratum": str(h),
                "footprint_stratum": str(f),
                "n": len(sub),
                "simulated_mean_eui": sim_mean,
                "measured_mean_eui": measured_mean,
                "bias_kwh_m2": sim_mean - measured_mean,
                "bias_pct": (sim_mean / measured_mean - 1) * 100,
                "median_floor_num": sub["floor_num"].median(),
                "median_footprint_m2": sub["footprint_area_m2"].median(),
            })

    bias_df = pd.DataFrame(bias_rows)
    bias_df.to_csv(OUTPUT_DIR / "bias_stratified_decomposition.csv", index=False, encoding="utf-8-sig")

    pivot_bias = bias_df.pivot(index="height_stratum", columns="footprint_stratum", values="bias_pct")
    pivot_n = bias_df.pivot(index="height_stratum", columns="footprint_stratum", values="n")

    fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.2), dpi=150, constrained_layout=True)
    im = axes[0].imshow(pivot_bias.values, cmap="RdBu_r", vmin=-30, vmax=30)
    axes[0].set_xticks(range(len(pivot_bias.columns)))
    axes[0].set_xticklabels(pivot_bias.columns)
    axes[0].set_yticks(range(len(pivot_bias.index)))
    axes[0].set_yticklabels(pivot_bias.index)
    axes[0].set_title("模拟 EUI 相对实测均值的分层偏差")
    for i in range(pivot_bias.shape[0]):
        for j in range(pivot_bias.shape[1]):
            val = pivot_bias.iloc[i, j]
            n_raw = pivot_n.iloc[i, j]
            if pd.isna(val) or pd.isna(n_raw):
                label = "NA"
            else:
                label = f"{val:+.1f}%\nn={int(n_raw)}"
            axes[0].text(j, i, label, ha="center", va="center", fontsize=8)
    cbar = fig.colorbar(im, ax=axes[0], shrink=0.86)
    cbar.set_label("偏差（%）")

    for label, sub in df_bias.groupby("height_stratum", observed=True):
        axes[1].hist(sub["eui_kwh_m2"], bins=22, alpha=0.48, density=True, label=str(label))
    axes[1].axvline(measured_mean, color="#B64646", linestyle="--", linewidth=1.4, label="实测均值（Chen et al., 2018）")
    axes[1].set_xlabel("建筑能源使用强度 EUI（kWh/(m2·a)）")
    axes[1].set_ylabel("概率密度")
    axes[1].set_title("不同高度分层的模拟 EUI 分布")
    axes[1].legend(frameon=False, fontsize=8)

    save_figure(fig, FIG_DIR / "bias_stratified_decomposition.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("偏差分层分解表已保存：", OUTPUT_DIR / "bias_stratified_decomposition.csv")
    display(bias_df.sort_values("bias_pct", ascending=False).round(3))
